<a href="https://colab.research.google.com/github/Alefros/alura-python-IA/blob/aula-3/AluraPythonIA_alunos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sorteio de alunos e distribuição de salas:

* 1 - Escolher uma ano entre 2000 e 2027
* 2 - Sortear um número de alunos entre 20 e 30 alunos
* 3 - Pedir para a IA uma lista com a quantidade de alunos sorteados acima, com base nos nomes comuns 10 anos antes do ano sorteado
* 4 - Distribuir os alunos em sala
* 5 - Imprimir os alunos
* 6 - Imprimir cada uma das salas  

In [143]:
#IMPORTS
import os
from google.colab import userdata
from google import genai

import random
import json
import math

from enum import Enum, IntEnum, StrEnum

#CONSTANTES
class ANO_LETIVO(IntEnum):
  INICIAL = 2000
  FINAL = 2027

class VAGAS_ABERTAS(Enum):
  MINIMO = 20
  MAXIMO = 45

class TAMANHO_DA_SALA(Enum):
  MINIMO = 7
  MAXIMO = 15

class GEMINI_MODELS(StrEnum):
  GEMINI_2_5_FLASH = "gemini-2.5-flash"
  GEMINI_1_5_FLASH_8B = "gemini-1.5-flash-8b"
  GEMINI_3_1_FLASH_LITE = "gemini-3.1-flash-lite"
  GEMINI_3_5_FLASH_LITE = "gemini-3.5-flash-lite"

USED_MODEL = GEMINI_MODELS.GEMINI_3_5_FLASH_LITE

IDADE_DE_INICIO = 10

In [144]:
#Preparar agente para perguntas
API_KEY = os.environ["GOOGLE_API_KEY"] = userdata.get('GERADOR_ALUNOS')
client = genai.Client()

In [145]:
# Sorteia um número em uma faixa
def sorteia_em_range(inicio, fim):
  numero_sorteado = random.choice(range(inicio, fim))
  return numero_sorteado

In [146]:
#Pedir para a IA uma lista de alunos
  #Gerar a lista com base nos nomes comuns 10 anos antes do ano sorteado
def matricula_de_alunos(ano, quantidade):

  prompt = (
      f"Gere um array JSON válido contendo exatamente {quantidade} strings com nomes e sobrenomes de crianças nascidas em {ano - IDADE_DE_INICIO}. "
      f"Exemplo de formato esperado: [\"Nome1 Sobrenome1\", \"Nome2 Sobrenome2\"]. "
      f"Atenção: retorne APENAS o JSON puro. Não inclua nenhuma explicação, formatação, introdução ou blocos de código markdown (como ```json)."
  )
  response = client.models.generate_content(model=USED_MODEL, contents=prompt)

  # Limpa possíveis blocos de código markdown (```json ... ```) se o modelo os gerar por engano
  texto_limpo = response.text.strip().strip('`').replace('json', '').strip()

  # Converte a string do JSON para uma lista real do Python
  try:
    lista_alunos = json.loads(texto_limpo)
    return lista_alunos
  except Exception as e:
    print(f"Erro ao converter o JSON: {e}")
    # Caso falhe por algum motivo, retorna a resposta em texto bruto
    return ""


In [162]:
#Sortear alunos por sala
def sorteia_sala(alunos, quantidade_de_alunos):
  sala_sorteada = []
  for rodada_do_sorteio in range(int(quantidade_de_alunos) + 1):
    aluno_sorteado = random.choice(alunos)
    sala_sorteada.append(aluno_sorteado)
    alunos.remove(aluno_sorteado)
  return sala_sorteada

In [148]:
#remover itens de uma lista
def remove_itens(lista, itens_a_remover):
  for item in itens_a_remover:
    lista.remove(str(item))
  return lista

In [149]:
#Imprimir lista de alunos com número
def imprimir_lista_de_alunos(lista_de_alunos, quantidade_de_alunos):
  numero_aluno = 1
  while numero_aluno <= quantidade_de_alunos:
    print(f"{numero_aluno} - " + lista_de_alunos[numero_aluno - 1])
    numero_aluno += 1


In [163]:
#MAIN

#Escolher uma ano entre 2000 e 2027
ano_sorteado = sorteia_em_range(ANO_LETIVO.INICIAL, ANO_LETIVO.FINAL)

#Sortear o número de alunos matriculados
quantidade_de_matriculados = sorteia_em_range(VAGAS_ABERTAS.MINIMO.value, VAGAS_ABERTAS.MAXIMO.value)

#Gerar uma lista de alunos com quantidade e nomes dinãmicos
alunos_matriculados = matricula_de_alunos(ano_sorteado, quantidade_de_matriculados)

#Gerar a quantidade de alunos em cada sala
tamanho_da_sala_maximo = sorteia_em_range(TAMANHO_DA_SALA.MINIMO.value, TAMANHO_DA_SALA.MAXIMO.value)
print("O número máximo de alunos é: " + str(tamanho_da_sala_maximo))

#Gerar a quantidade de salas
quantidade_de_salas = math.ceil(quantidade_de_matriculados / tamanho_da_sala_maximo)

#Imprimir os alunos
print("Para o ano de " + str(ano_sorteado) +
        " recebemos a matrícula de " + str(quantidade_de_matriculados) +
        " alunos, que foram distribuídos em " + str(quantidade_de_salas) +
        " salas, são eles: \n")

imprimir_lista_de_alunos(alunos_matriculados, quantidade_de_matriculados)
print("\n")

#Distribuir os alunos em sala
sala = 1
while sala <= quantidade_de_salas:
  if sala == quantidade_de_salas:
    sala_sorteada = alunos_matriculados.copy()
  else:
    sala_sorteada = sorteia_sala(alunos_matriculados.copy(), math.floor(quantidade_de_matriculados / quantidade_de_salas))
    alunos_matriculados = remove_itens(alunos_matriculados, sala_sorteada)

  #ordenar a sala
  sala_sorteada.sort()

  #Imprimir cada uma das salas
  print("Sala " + str(sala) + ":")
  imprimir_lista_de_alunos(sala_sorteada, len(sala_sorteada))
  print("\n")
  sala += 1

O número máximo de alunos é: 11
Para o ano de 2023 recebemos a matrícula de 20 alunos, que foram distribuídos em 2 salas, são eles: 

1 - Lucas Silva
2 - Beatriz Santos
3 - Enzo Oliveira
4 - Valentina Souza
5 - Gabriel Rodrigues
6 - Júlia Ferreira
7 - Pedro Lima
8 - Sophia Alves
9 - Matheus Pereira
10 - Laura Ribeiro
11 - Lucas Carvalho
12 - Isis Gomes
13 - Guilherme Martins
14 - Heloísa Rocha
15 - Davi Dias
16 - Clara Castro
17 - Arthur Araujo
18 - Alice Cardoso
19 - Bernardo Pires
20 - Manuela Correia


Sala 1:
1 - Alice Cardoso
2 - Arthur Araujo
3 - Davi Dias
4 - Gabriel Rodrigues
5 - Guilherme Martins
6 - Isis Gomes
7 - Laura Ribeiro
8 - Lucas Carvalho
9 - Manuela Correia
10 - Matheus Pereira
11 - Pedro Lima


Sala 2:
1 - Beatriz Santos
2 - Bernardo Pires
3 - Clara Castro
4 - Enzo Oliveira
5 - Heloísa Rocha
6 - Júlia Ferreira
7 - Lucas Silva
8 - Sophia Alves
9 - Valentina Souza


